# Lab 01 — Anatomy of Real Cosmological Data

**Course:** MSc Astrostatistics for Observational Cosmology  
**Format:** 4-hour hands-on practical  
**Status:** Formative + weekly checkpoint  
**Real data:** Pantheon+ / SH0ES and SDSS Reverberation Mapping (SDSS-RM)

> ## Scientific question
> **What was actually measured, what was derived, and where does uncertainty enter before we do any cosmological inference?**

This first lab is deliberately about **data literacy before model fitting**. You will work with two real observational datasets:

1. a Type-Ia-supernova Hubble-diagram product from Pantheon+ / SH0ES;
2. continuum and broad-line light-curve products from SDSS-RM.

You will **not fit cosmological parameters today**.

---

## Learning outcomes

By the end of the lab you should be able to:

- distinguish a physical source from a catalogue row;
- distinguish measured, derived, calibration and model quantities;
- identify units, flags, missing values and repeated observations;
- understand why a published uncertainty column may be valid for plotting but not for inference;
- inspect an irregular AGN light curve without touching a spectrum;
- state a statistical question in terms of an **estimand**, **observations**, assumptions and validation.

## 0. Before coding — the five-line statement

Write one sentence for each item **before running the scientific analysis**.

1. **Estimand:** What quantity would you eventually like to infer?
2. **Observed data:** Which quantities were measured or reported?
3. **Generative assumptions:** What physical/statistical chain might connect the estimand to the observations?
4. **Uncertainty / selection:** What could distort that connection?
5. **Validation:** What observation or diagnostic could make you distrust the analysis?

### Your answer

**Estimand:**  
**Observed data:**  
**Generative assumptions:**  
**Uncertainty / selection:**  
**Validation:**

## 1. Setup

Run the next cell. The notebook uses only standard scientific-Python packages plus `astroquery` for the SDSS-RM part.

If you are using Google Colab and `astroquery` is missing, run:

```python
!pip -q install astroquery
```

In [ ]:
from pathlib import Path
import io, sys, warnings, requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from astropy.cosmology import FlatLambdaCDM

SEED = 2026
rng = np.random.default_rng(SEED)

pd.set_option("display.max_columns", 60)
print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

# Part I — Pantheon+ / SH0ES

## 2. Download a real public data product

We use the published file `Pantheon+SH0ES.dat`.

Important terminology from the data release:

- `CID`: candidate/SN identifier;
- `IDSURVEY`: survey identifier;
- `zHD`: Hubble-diagram redshift;
- `m_b_corr`: corrected/standardized \(m_B\);
- `m_b_corr_err_DIAG`: diagonal uncertainty for visualization;
- `MU_SH0ES`: standardized distance modulus using the SH0ES Cepheid calibration;
- `MU_SH0ES_ERR_DIAG`: diagonal uncertainty for visualization;
- `IS_CALIBRATOR`: whether the SN host has a Cepheid distance.

**Critical warning from the Pantheon+ release:** the diagonal uncertainty columns are for plotting/visualization. Cosmological inference requires the released covariance matrix. We will exploit that distinction later in the course.

In [ ]:
PANTHEON_URL = (
    "https://raw.githubusercontent.com/PantheonPlusSH0ES/DataRelease/main/"
    "Pantheon%2B_Data/4_DISTANCES_AND_COVAR/Pantheon%2BSH0ES.dat"
)

def load_pantheon(url=PANTHEON_URL):
    """Download the published Pantheon+SH0ES high-level table."""
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    return pd.read_csv(io.StringIO(r.text), sep=r"\s+")

sn = load_pantheon()
print(f"Rows: {len(sn):,}")
print(f"Unique CID values: {sn['CID'].nunique():,}")
display(sn.head())

## 3. First audit: never analyse a table you have not inspected

### Exercise 1

Without making a cosmological plot yet:

1. print the column names;
2. report the number of rows;
3. report the number of unique `CID` values;
4. find how many `CID`s occur more than once;
5. inspect the five most frequently repeated `CID`s;
6. check for non-finite values in `zHD`, `MU_SH0ES`, `m_b_corr`.

**Question:** Why can the number of catalogue rows exceed the number of physical SNe?

In [ ]:
# TODO: complete the audit.
print(sn.columns.tolist())

counts_per_cid = sn.groupby("CID").size().sort_values(ascending=False)

# TODO:
# n_rows = ...
# n_unique = ...
# n_repeated_cids = ...
# display(counts_per_cid.head())

### Interpretation box

Write 3–5 sentences. In particular, explain why treating every row as an independent physical SN can be dangerous in a resampling exercise.

## 4. Survey composition

Pantheon+ combines multiple surveys. The release maps `IDSURVEY` values to survey names.

For this lab, use the following subset of the published mapping:

In [ ]:
SURVEY_MAP = {
    1:"SDSS", 4:"SNLS", 5:"CSP", 10:"DES", 15:"PS1MD",
    18:"CNIa0.02", 50:"LOWZ/JRK07", 51:"LOSS1", 56:"SOUSA",
    57:"LOSS2", 61:"CFA1", 62:"CFA2", 63:"CFA3S", 64:"CFA3K",
    65:"CFA4p2", 66:"CFA4p3", 100:"HST", 101:"SNAP",
    106:"CANDELS", 150:"FOUND"
}
sn["SURVEY"] = sn["IDSURVEY"].map(SURVEY_MAP).fillna("OTHER")

survey_counts = sn["SURVEY"].value_counts()
display(survey_counts.head(12))

ax = survey_counts.head(12).sort_values().plot(kind="barh")
ax.set_xlabel("Number of catalogue rows")
ax.set_ylabel("Survey")
ax.set_title("Pantheon+ survey composition")
plt.show()

### Exercise 2

Answer:

1. Is Pantheon+ a single homogeneous survey?
2. Why might survey identity matter statistically even after standardization?
3. Which later course topic is already visible here: independence, selection, covariance, domain shift, or all of them?

## 5. Redshift and uncertainty structure

Use `zHD` for the Hubble-diagram redshift.

For **visualization only**, inspect `MU_SH0ES_ERR_DIAG`.

In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
ax.hist(sn["zHD"], bins=50)
ax.set_xlabel(r"$z_{\rm HD}$")
ax.set_ylabel("Number of rows")
ax.set_title("Pantheon+ Hubble-diagram redshift distribution")
plt.show()

fig, ax = plt.subplots(figsize=(7,4))
ax.scatter(sn["zHD"], sn["MU_SH0ES_ERR_DIAG"], s=8, alpha=0.5)
ax.set_xlabel(r"$z_{\rm HD}$")
ax.set_ylabel(r"Diagonal uncertainty in $\mu$ [mag]")
ax.set_title("Published diagonal uncertainty versus redshift")
plt.show()

### Exercise 3 — do not just describe the picture

Calculate for three redshift intervals:

\[
z<0.1,\qquad 0.1\le z<0.5,\qquad z\ge 0.5,
\]

the number of rows, median diagonal uncertainty, and MAD of the diagonal uncertainty.

Then answer:

> Does the measurement precision remain constant with redshift?

Do **not** infer a causal explanation from this table alone.

In [ ]:
bins = pd.cut(
    sn["zHD"],
    bins=[-np.inf, 0.1, 0.5, np.inf],
    labels=["z<0.1", "0.1<=z<0.5", "z>=0.5"]
)

def mad(x):
    x = np.asarray(x, float)
    med = np.nanmedian(x)
    return np.nanmedian(np.abs(x-med))

summary = (
    sn.assign(zbin=bins)
      .groupby("zbin", observed=True)["MU_SH0ES_ERR_DIAG"]
      .agg(N="size", median="median", MAD=mad)
)
display(summary)

## 6. Observed, derived, calibrated or model quantity?

Complete this table in a Markdown cell.

| Quantity | Meaning | Directly observed? | Derived/calibrated? | Used as model parameter? |
|---|---|---|---|---|
| `zHEL` |  |  |  |  |
| `zHD` |  |  |  |  |
| `mB` |  |  |  |  |
| `m_b_corr` |  |  |  |  |
| `MU_SH0ES` |  |  |  |  |
| \(H_0\) |  |  |  |  |
| \(\Omega_m\) |  |  |  |  |

### Key question

Why is `MU_SH0ES` **not** simply a raw telescope measurement?

# Part II — SDSS Reverberation Mapping

## 7. Retrieve published SDSS-RM light-curve products

The VizieR catalogue `J/ApJ/851/21` contains:

- `table1`: 222 quasar records;
- `table2`: 152,258 continuum/emission-line measurements;
- `table4`: 44 H\(\beta\) reverberation-mapped objects;
- `table5`: 18 H\(\alpha\) reverberation-mapped objects.

The light-curve table contains bands `g`, `i`, `Halpha`, and `Hbeta`.

The published `MJD` field is **MJD − 50000**, so we restore the full MJD for plotting.

In [ ]:
try:
    from astroquery.vizier import Vizier
except ImportError:
    raise ImportError("Install astroquery first: pip install astroquery")

Vizier.ROW_LIMIT = -1

sample_tab = Vizier.get_catalogs("J/ApJ/851/21/table1")[0].to_pandas()
lc_tab = Vizier.get_catalogs("J/ApJ/851/21/table2")[0].to_pandas()
hb_tab = Vizier.get_catalogs("J/ApJ/851/21/table4")[0].to_pandas()

for frame in (sample_tab, lc_tab, hb_tab):
    for col in frame.select_dtypes(include=["object"]).columns:
        frame[col] = frame[col].astype(str).str.strip()

lc_tab["MJD_full"] = lc_tab["MJD"].astype(float) + 50000.0

print("SDSS-RM sample rows:", len(sample_tab))
print("SDSS-RM light-curve rows:", len(lc_tab))
print("Published Hbeta-lag objects:", len(hb_tab))
display(sample_tab.head())
display(lc_tab.head())

## 8. Inspect the data model

### Exercise 4

For `table2`, identify:

- object identifier;
- time;
- observed band;
- telescope;
- continuum flux and its uncertainty;
- integrated line flux and its uncertainty;
- rejection flag.

Then calculate how many rows belong to each `Band`.

**Question:** Why is a random train/test split by rows invalid if we later build a machine-learning model of objects?

In [ ]:
display(lc_tab["Band"].value_counts())

# TODO:
# How many distinct RMID objects occur in the full light-curve table?
# How many observations does the median object have?

## 9. Plot one real AGN without touching a spectrum

We choose the first object in the published H\(\beta\)-lag table. This is not because it is "best"; it simply guarantees that a published H\(\beta\) lag exists for later comparison.

In [ ]:
example_rmid = int(hb_tab.iloc[0]["RMID"])
obj = lc_tab[lc_tab["RMID"].astype(int) == example_rmid].copy()
z_obj = float(sample_tab.loc[sample_tab["RMID"].astype(int)==example_rmid, "z"].iloc[0])

g = obj[obj["Band"]=="g"].copy()
hb = obj[(obj["Band"]=="Hbeta") & (obj["Flag"].fillna(0).astype(int)==0)].copy()

print("Example RMID:", example_rmid)
print("Redshift:", z_obj)
print("g-band rows:", len(g), "| Hbeta rows:", len(hb))

fig, axes = plt.subplots(2,1, figsize=(9,6), sharex=True)

axes[0].errorbar(
    g["MJD_full"], g["Flux"], yerr=g["e_Flux"],
    fmt=".", ms=4, capsize=2
)
axes[0].set_ylabel("g continuum flux")
axes[0].set_title(f"SDSS-RM {example_rmid}: real published light curves")

axes[1].errorbar(
    hb["MJD_full"], hb["S"], yerr=hb["e_S"],
    fmt=".", ms=4, capsize=2
)
axes[1].set_xlabel("MJD")
axes[1].set_ylabel(r"H$\beta$ integrated flux")

plt.tight_layout()
plt.show()

## 10. Cadence is part of the data

For the \(g\)-band continuum and H\(\beta\) line separately, calculate:

- number of epochs;
- observing baseline;
- median separation between adjacent epochs;
- minimum and maximum separation.

### Why this matters

A lag is not inferred from \(C(t)\) and \(L(t)\) in the abstract. It is inferred from **irregularly sampled measurements** of them.

In [ ]:
def cadence_summary(time):
    t = np.sort(np.asarray(time, dtype=float))
    dt = np.diff(t)
    return pd.Series({
        "N_epochs": len(t),
        "baseline_days": t.max()-t.min() if len(t)>1 else np.nan,
        "median_dt_days": np.median(dt) if len(dt)>0 else np.nan,
        "min_dt_days": np.min(dt) if len(dt)>0 else np.nan,
        "max_dt_days": np.max(dt) if len(dt)>0 else np.nan,
    })

cadence = pd.DataFrame({
    "g continuum": cadence_summary(g["MJD_full"]),
    "Hbeta": cadence_summary(hb["MJD_full"])
}).T

display(cadence)

# 11. Checkpoint

Show the instructor/assistant:

1. Pantheon+ survey-composition plot;
2. redshift–uncertainty plot;
3. your observed/derived/calibrated/model table;
4. one real SDSS-RM continuum + H\(\beta\) light-curve figure;
5. the cadence table.

### Exit question

In no more than 120 words:

> Why is “download catalogue → fit model” not an adequate statistical workflow?

# 12. Preparation for next week

Next week we remain with Pantheon+, but move from **data anatomy** to:

- Hubble residuals;
- robust summaries;
- bootstrap uncertainty;
- permutation testing;
- empirical CDFs.

You will still **not** use the diagonal Pantheon+ error column for cosmological parameter fitting. The full covariance enters later.